# 01 — Exploração dos dados (CGMacros)
**Integrante 1 — Pré-processamento e análise dos dados**

Objetivo: conhecer o dataset *antes* de tratá-lo (Passo 1 do roteiro Data-Centric da aula: "baixar e entender os dados").
Perguntas que este notebook responde:
1. Quais arquivos e colunas existem? Batem com o dicionário de dados?
2. Quantos dias cada participante tem? Os dias começam à meia-noite?
3. Quanto dado falta (glicose, HR, METs)? Há valores impossíveis ou duplicados?
4. Como são as refeições e a atividade?
5. A glicose muda entre os grupos (saudável / pré-diabetes / diabetes tipo 2)?

> As decisões de limpeza tomadas a partir daqui ficam registradas no notebook `02_preprocessamento.ipynb` e em `preprocessing.py`.

## 0. Configuração

In [ ]:
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append("..")           # para importar preprocessing.py
import preprocessing as P

# >>> Ajuste aqui a pasta onde você baixou o CGMacros (a busca é recursiva) <<<
RAW_DIR = Path(os.environ.get("CGMACROS_RAW", "../data/raw"))

pd.set_option("display.max_columns", 40, "display.width", 160)
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True})
print("Procurando dados em:", RAW_DIR.resolve())

## 1. Arquivos disponíveis

In [ ]:
files = P.find_participant_files(RAW_DIR)
bio_path = P.find_bio_file(RAW_DIR)
print(f"{len(files)} arquivos de participantes | bio.csv: {bio_path}")
print("IDs:", sorted(files))
print("IDs ausentes entre 1 e", max(files), ":", sorted(set(range(1, max(files) + 1)) - set(files)))
print("Tamanho médio dos CSVs (MB):", round(np.mean([f.stat().st_size for f in files.values()]) / 1e6, 2))

### Os arquivos NÃO têm todos as mesmas colunas
Lemos só o cabeçalho de cada CSV. No dataset real há variações importantes: alguns participantes têm `Intensity` (indicador de intensidade por minuto; no ID 27 só aparecem 0 e 1) no lugar de `METs`,
o participante 7 tem `Steps` e não tem `Calories (Activity)`, alguns não têm `Amount Consumed`, um tem `Sugar`, alguns têm coluna de índice, e a ordem Libre/Dexcom varia.

In [ ]:
rows = []
for pid, f in files.items():
    n = [P._norm(c) for c in pd.read_csv(f, nrows=0).columns]
    rows.append({"participant_id": pid, "tem_METs": "mets" in n, "tem_Intensity": "intensity" in n, "tem_Steps": "steps" in n,
                 "tem_Calories_Activity": "calories (activity)" in n, "tem_Amount_Consumed": "amount consumed" in n,
                 "tem_Sugar": "sugar" in n, "coluna_indice": n[0].startswith("unnamed"),
                 "dexcom_antes_libre": ("dexcom gl" in n and "libre gl" in n and n.index("dexcom gl") < n.index("libre gl"))})
hdr_df = pd.DataFrame(rows).set_index("participant_id")
display(hdr_df.sum().to_frame("n_participantes"))
print("Com Intensity (sem METs):", hdr_df.index[hdr_df["tem_Intensity"]].tolist())
print("Sem Calories (Activity):", hdr_df.index[~hdr_df["tem_Calories_Activity"]].tolist())

## 2. Um participante em detalhe
Primeiro com os dados **crus** (só renomeei as colunas para um padrão). O dicionário de dados diz:
`Timestamp, Libre GL, Dexcom GL, HR, Calories (Activity), Mets (×10), Meal Type, Calories, Carbs, Protein, Fat, Fiber, Amount Consumed, Image Path`.

In [ ]:
pid0 = sorted(files)[0]
raw0 = P.load_participant_csv(files[pid0])
print("Participante", pid0, "| linhas:", len(raw0))
print("Colunas:", list(raw0.columns))
raw0.head()

In [ ]:
raw0.info()

In [ ]:
# Frequência de amostragem: diferença entre timestamps consecutivos
dt = raw0["timestamp"].diff().dropna().dt.total_seconds() / 60
print(dt.value_counts().head(5))
print("Início:", raw0["timestamp"].min(), "| Fim:", raw0["timestamp"].max(),
      "| Duração (dias):", round((raw0["timestamp"].max() - raw0["timestamp"].min()).total_seconds() / 86400, 2))

In [ ]:
# Linhas em que há refeição (Meal Type preenchido): é aí que ficam os macros
raw0[raw0["meal_type"].notna()].head(10)

## 3. Duração e dias de cada participante
**Ponto crítico para o projeto:** precisamos de D1…D9. O registro começa no meio do dia, então o "D1" por data de calendário é parcial (~11 datas de calendário para ~10 dias de dados). Veja o histograma da hora de início.

In [ ]:
rows = []
raws = {}
for pid, f in files.items():
    r = P.load_participant_csv(f); raws[pid] = r
    t = r["timestamp"]
    rows.append({"participant_id": pid, "rows": len(r), "start": t.min(), "end": t.max(),
                 "duration_days": (t.max() - t.min()).total_seconds() / 86400,
                 "calendar_days": t.dt.normalize().nunique(),
                 "start_hour": t.min().hour + t.min().minute / 60,
                 "duplicated_ts": int(t.duplicated().sum())})
dur = pd.DataFrame(rows).set_index("participant_id")
display(dur.describe().T[["min", "mean", "max"]])
dur.head()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
dur["duration_days"].hist(ax=ax[0], bins=20); ax[0].set_title("Duração do registro (dias)")
dur["start_hour"].hist(ax=ax[1], bins=24); ax[1].set_title("Hora do dia em que o registro começa")
plt.show()
print("Participantes com menos de 9 dias de calendário:", dur.index[dur["calendar_days"] < 9].tolist())

## 4. Dados faltantes e valores impossíveis
Na aula: "muitos datasets possuem valores não preenchidos". Aqui medimos *quanto* falta em cada coluna, por participante.

In [ ]:
cols = ["libre_gl", "dexcom_gl", "hr", "cal_activity", "mets"]
miss = pd.DataFrame({pid: r[cols].isna().mean() * 100 for pid, r in raws.items()}).T
miss.index.name = "participant_id"
display(miss.describe().T[["min", "mean", "max"]].round(1))
fig, ax = plt.subplots(figsize=(11, 4))
miss.plot(kind="box", ax=ax); ax.set_ylabel("% de linhas faltantes"); ax.set_title("Dados faltantes por coluna (cada ponto = 1 participante)")
plt.show()

In [ ]:
# Glicose fora do intervalo físico do sensor (40-400 mg/dL) e timestamps duplicados
oor = {pid: int(((r[["libre_gl", "dexcom_gl"]] < P.GLUCOSE_MIN) | (r[["libre_gl", "dexcom_gl"]] > P.GLUCOSE_MAX)).sum().sum())
       for pid, r in raws.items()}
print("Leituras de glicose fora de [40, 400]:", sum(oor.values()), "| participantes afetados:", sum(v > 0 for v in oor.values()))
print("Timestamps duplicados no total:", int(dur["duplicated_ts"].sum()))

## 5. Glicose: Libre × Dexcom
Temos dois sensores medindo a mesma coisa. **No dataset real o Dexcom lê ~32 mg/dL acima do Libre** (e a correlação minuto a minuto é só ~0,83).
Qual está certo? A seção 10 compara os dois com as pontas de dedo e com a A1c. Nas 32 pontas de dedo em que os dois sensores já estavam ativos, o Libre ficou **15,9 mg/dL abaixo**
e o Dexcom **16,8 mg/dL acima** (erro absoluto médio ≈ 21 e 20); a **média dos dois** ficou a +0,5 mg/dL (erro absoluto médio 9,6). Por isso o padrão do pipeline é o `consensus` (média alinhada por participante).
Se a checagem com a A1c (seção 10) contradisser isso, troque `glucose_source`.

In [ ]:
both = pd.concat([r[["libre_gl", "dexcom_gl"]].dropna() for r in raws.values()])
print("Pares simultâneos:", len(both), "| correlação:", round(both.corr().iloc[0, 1], 3),
      "| diferença média (Dexcom - Libre):", round((both["dexcom_gl"] - both["libre_gl"]).mean(), 2), "mg/dL")
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
s = both.sample(min(5000, len(both)), random_state=0)
ax[0].scatter(s["libre_gl"], s["dexcom_gl"], s=3, alpha=.3); ax[0].plot([40, 400], [40, 400], "r--")
ax[0].set_xlabel("Libre"); ax[0].set_ylabel("Dexcom"); ax[0].set_title("Concordância entre sensores")
both.plot(kind="hist", bins=60, alpha=.6, ax=ax[1]); ax[1].set_title("Distribuição da glicose (mg/dL)")
plt.show()

## 6. Refeições

In [ ]:
meal_raw = pd.concat([r.assign(participant_id=pid) for pid, r in raws.items()])
meal_raw = meal_raw[meal_raw["meal_type"].notna()].copy()
print("Valores crus de Meal Type:", meal_raw["meal_type"].value_counts().to_dict())
meal_raw["meal_type"] = P.normalize_meal_type(meal_raw["meal_type"])   # snack, snacks, snack 1 -> snack
meal_raw["day_cal"] = meal_raw["timestamp"].dt.normalize()
print("Total de linhas com refeição:", len(meal_raw))
print(meal_raw["meal_type"].value_counts())
display(meal_raw[list(P.NUTRIENTS) + ["amount_consumed"]].describe().T.round(1))

In [ ]:
# CHECAGEM IMPORTANTE: cada refeição aparece em 1 linha ou em 2 (início e fim)?
per = meal_raw.groupby(["participant_id", "day_cal", "meal_type"]).size().rename("linhas").reset_index()
print("Linhas por (participante, dia, tipo) - REFEICOES PRINCIPAIS:")
print(per[per["meal_type"].isin(P.MEAL_TYPES)]["linhas"].value_counts().sort_index())
print("Linhas por (participante, dia) - LANCHES (varios por dia e normal):")
print(per[per["meal_type"] == "snack"]["linhas"].value_counts().sort_index())
print("-> nas refeicoes principais, 1 dominante = uma linha por refeicao (marca o inicio); 2 seria inicio+fim.")

In [ ]:
# Valores implausiveis (erros de digitacao): comparar com os tetos de preprocessing.MEAL_VALUE_CAPS
print("Tetos por refeicao:", P.MEAL_VALUE_CAPS)
print({c: int((meal_raw[c] > cap).sum()) for c, cap in P.MEAL_VALUE_CAPS.items()}, "valores acima do teto")
print("Amount Consumed > 100:", int((meal_raw["amount_consumed"] > 100).sum()))
display(meal_raw.nlargest(8, "fiber")[["participant_id", "timestamp", "meal_type", "calories", "carbs", "protein", "fat", "fiber"]])
fig, ax = plt.subplots(1, 4, figsize=(14, 3))
for a, c in zip(ax, ["calories", "carbs", "protein", "fat"]):
    meal_raw[c].hist(bins=30, ax=a); a.set_title(c)
plt.show()
meal_raw.groupby("meal_type")[list(P.NUTRIENTS)].mean().round(1)

## 7. Atividade física (HR e METs)

Só quem tem a coluna `METs` entra no histograma de METs abaixo. Para quem tem `Intensity` (sem METs), o pipeline **estima** os METs a partir de `Calories (Activity)`
(o gasto por minuto é proporcional aos METs; 1 MET = gasto mínimo do participante). Conferi em arquivos reais: no ID 1 (que tem `METs`), `Calories (Activity)` = 1,0484 × METs em todos os minutos;
no ID 27 (que tem `Intensity`), as calorias são múltiplos exatos de 1,0884 (o gasto em repouso), como esperado para METs×10 inteiros. A seção 9 repete essa validação em todos os participantes que têm `METs`.

In [ ]:
com_mets = hdr_df.index[hdr_df["tem_METs"]]
act = pd.concat([raws[p][["hr", "mets", "cal_activity"]] for p in com_mets])
act = act.assign(mets=act["mets"] / 10)   # o arquivo guarda METs x 10
display(act.describe().T.round(2))
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
act["hr"].hist(bins=60, ax=ax[0]); ax[0].set_title("HR (bpm)")
act["mets"].clip(upper=12).hist(bins=60, ax=ax[1]); ax[1].set_title("METs (÷10)")
plt.show()

## 8. Dados dos participantes (`bio.csv`)
Atenção ao dicionário: LDL=800 e VLDL=400 são **erros de cálculo**, e a A1c está documentada como mmol/mol mas os valores (4.6–8.5) são claramente %.

In [ ]:
bio_raw = pd.read_csv(bio_path)
print(bio_raw.shape); display(bio_raw.head())
bio = P.load_bio(bio_path)
print("\nApós limpeza:"); display(bio.describe().T[["count", "mean", "min", "max"]].round(2))
print(bio["group_a1c"].value_counts())

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.2))
bio["a1c"].hist(bins=20, ax=ax[0]); ax[0].axvline(5.7, c="r"); ax[0].axvline(6.5, c="r"); ax[0].set_title("A1c (%) — cortes 5.7 e 6.5")
bio["bmi"].hist(bins=20, ax=ax[1]); ax[1].set_title("IMC")
bio["age"].hist(bins=20, ax=ax[2]); ax[2].set_title("Idade")
plt.show()
print("Participantes no bio sem arquivo CGM:", sorted(set(bio.participant_id) - set(files)))
print("Arquivos CGM sem linha no bio:", sorted(set(files) - set(bio.participant_id)))

## 9. Perfil glicêmico por grupo (já com limpeza básica)
Aqui já uso `clean_participant` (detalhado no notebook 02) só para comparar os grupos.

In [ ]:
minutes = []
for pid, r in raws.items():
    m, _ = P.clean_participant(r, pid); minutes.append(m)
minute = pd.concat(minutes, ignore_index=True)
minute = minute.merge(bio[["participant_id", "group_a1c"]], on="participant_id", how="left")
print(minute.shape)

### Validação do estimador de METs
Em quem TEM `METs`, comparamos o valor real com o estimado só pelas calorias (o que o pipeline faz para quem só tem `Intensity`).

In [ ]:
rows = []
for pid in hdr_df.index[hdr_df["tem_METs"]][:20]:
    r = raws[pid]; cal = r["cal_activity"]; true = r["mets"] / 10
    est = cal / cal[cal > 0].quantile(0.01)
    ok = true.notna() & est.notna()
    rows.append({"participant_id": pid, "corr": true[ok].corr(est[ok]), "METs_real_medio": true[ok].mean(), "METs_estimado_medio": est[ok].mean(),
                 "%ativo_real": 100 * (true[ok] >= P.ACTIVE_METS).mean(), "%ativo_estimado": 100 * (est[ok] >= P.ACTIVE_METS).mean()})
display(pd.DataFrame(rows).round(2))

In [ ]:
order = ["healthy", "prediabetes", "t2d"]
prof = minute.assign(hour=minute["minute_of_day"] // 60).groupby(["group_a1c", "hour"])["glucose"].mean().unstack(0)[order]
prof.plot(figsize=(10, 4)); plt.xlabel("Hora do dia"); plt.ylabel("Glicose média (mg/dL)"); plt.title("Perfil médio de 24 h por grupo"); plt.show()
daily_g = minute.groupby(["participant_id", "group_a1c"])["glucose"].agg(["mean", "std"]).reset_index()
daily_g.boxplot(column="mean", by="group_a1c", figsize=(6, 4)); plt.suptitle(""); plt.title("Glicose média por participante"); plt.show()

## 10. Validação dos sensores: ponta de dedo e A1c
O `bio.csv` traz 3 glicemias de ponta de dedo (com horário) na manhã do 1º dia, logo depois de o sensor ser colocado. Comparamos os sensores com elas
(`fingerstick_validation`). **Cuidados:** são poucas medições, no início do uso do sensor (aquecimento) e às vezes durante variações rápidas de glicose; serve para ver a *ordem de grandeza* do viés.
`midpoint` = (Libre + Dexcom)/2; `pipeline_glucose` = a série que o pipeline usa.

In [ ]:
det, summ = P.fingerstick_validation(minute, bio)
print(f"{len(det)} medições de ponta de dedo com Libre e Dexcom disponíveis ({det['participant_id'].nunique()} participantes)")
display(summ.round(1))
fig, ax = plt.subplots(figsize=(5.5, 5))
for c in ("libre", "dexcom", "midpoint"):
    ax.scatter(det["fingerstick"], det[c], s=14, alpha=.6, label=c)
lim = [det["fingerstick"].min() - 10, det["fingerstick"].max() + 10]
ax.plot(lim, lim, "k--"); ax.set_xlabel("ponta de dedo (mg/dL)"); ax.set_ylabel("sensor (mg/dL)"); ax.legend(); plt.show()

Segunda checagem, sem depender do horário: a A1c permite estimar a glicose média esperada (fórmula ADAG: `eAG = 28.7 × A1c − 46.7`).
Comparamos a média de cada sensor, por participante, com essa estimativa.

In [ ]:
eag = bio.set_index("participant_id")["a1c"] * 28.7 - 46.7
chk = minute.groupby("participant_id")[["libre_gl", "dexcom_gl", "glucose"]].mean()
chk["eAG_da_A1c"] = eag
chk = chk.dropna()
for c in ("libre_gl", "dexcom_gl", "glucose"):
    print(f"{c:10s} viés médio vs eAG = {(chk[c] - chk['eAG_da_A1c']).mean():+6.1f} mg/dL | correlação = {chk[c].corr(chk['eAG_da_A1c']):.2f}")
fig, ax = plt.subplots(figsize=(5.5, 5))
for c in ("libre_gl", "dexcom_gl", "glucose"):
    ax.scatter(chk["eAG_da_A1c"], chk[c], s=14, alpha=.7, label=c)
ax.plot([90, 220], [90, 220], "k--"); ax.set_xlabel("glicose média estimada pela A1c"); ax.set_ylabel("média do sensor"); ax.legend(); plt.show()

## 11. Conclusões e decisões (preencher com o grupo)
Responda olhando as saídas acima — essas decisões vão para o relatório/apresentação:
- **Dias:** o registro começa no meio do dia (em geral ~9–10 h), então o 1º dia de calendário é parcial. Por isso o padrão é `day_mode="first_full"`: D1 = primeiro dia com glicose suficiente, o que deixa D1–D9 completos (cabem 9 dias inteiros em ~10 dias de registro). Quantos participantes têm D1–D9?
- **Glicose:** os sensores não concordam (Dexcom ~32 mg/dL acima do Libre) e, contra a ponta de dedo (n=32), o Libre lê 15,9 abaixo e o Dexcom 16,8 acima; a média dos dois erra +0,5 (MAE 9,6 contra ~21 de cada sensor sozinho). Pela A1c (seção 10), o viés médio foi −20,5 (Libre), +11,5 (Dexcom) e −4,3 mg/dL (média), com correlação 0,81/0,81/0,85. As duas checagens concordam: o padrão é `consensus` (média alinhada por participante); `glucose_source` permite trocar.
- **Atividade:** 11 participantes têm `Intensity` no lugar de `METs`; os METs desses são estimados pelas calorias (ver validação na seção 9). O ID 7 tem `METs`, mas não tem `Calories (Activity)`.
- **Colunas:** os arquivos têm cabeçalhos diferentes (coluna de índice, ordem trocada, sem `Amount Consumed`, `Sugar`); o leitor trata tudo por nome.
- **Refeições:** café/almoço/jantar ocupam 1 linha cada (marcam o início). Há lanches com três grafias ('snack', 'snacks', 'snack 1'), unificados em `snack`; vários por dia são normais.
- **Erros de digitação nos macros** (ex.: fibra de 2830 g, carboidrato de 761 g): valores acima de `MEAL_VALUE_CAPS` viram faltantes e são imputados pela mediana do mesmo tipo de refeição do participante.
- **Missing:** quais colunas/participantes têm mais falta? Algum participante deve ser excluído?
- **Valores-erro:** LDL=800, VLDL=400, glicose fora de 40–400 → viram `NaN` (já tratado em `preprocessing.py`).